# Section Estimators with `Scikit-Learn`

The task is to estimate the properties of a particular cross section.

In [1]:
from typing import Optional

import pandas as pd
from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_val_score,
    RandomizedSearchCV,
)
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.compose import TransformedTargetRegressor
from sklearn.base import BaseEstimator
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
    median_absolute_error,
)
import numpy as np
import mlflow
import json, os

from cso import (
    CANONICAL_SCORE_NAME,
    CROSS_SECTION_PARAMETERS,
    MATERIAL_PARAMETERS,
    print_system_info,
)
from cso.ml import canonical_regression_score

print_system_info()

Python version: 3.12.7 (v3.12.7:0b05ead877f, Sep 30 2024, 23:18:00) [Clang 13.0.0 (clang-1300.0.29.30)]
Operating System: Darwin 23.4.0
Platform: macOS-14.4-arm64-arm-64bit
Processor: arm
Machine: arm64
CPU count: 11


In [2]:
config_file_path = "../fixtures/config_I.json"
data_file_path = "../fixtures/section_data_I_10000.csv"
mlflow_tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "sqlite:///mlflow.db")
mlflow_experiment_name=None
task = "section_estimation"

In [3]:
with open(config_file_path, "r") as f:
    config: dict = json.load(f)

In [4]:
# Load section data
section_data = config["section"]

section_type = config["section"]["type"]
print(f"Section type from config: {section_type}")

section_variables = []
for p in section_data["params"].keys():
    if section_data["params"][p]["variable"]:
        section_variables.append(p)

predictor_columns = section_variables + MATERIAL_PARAMETERS
target_columns = CROSS_SECTION_PARAMETERS

# Initialize MLflow run
mlflow_experiment_name = mlflow_experiment_name or task
mlflow.set_tracking_uri(mlflow_tracking_uri)
experiment = mlflow.set_experiment(mlflow_experiment_name)
mlflow.set_experiment_tag("task", task)
mlflow.sklearn.autolog(log_models=False)

Section type from config: i_section


2025/11/16 16:55:07 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/11/16 16:55:07 INFO mlflow.store.db.utils: Updating database tables
2025-11-16 16:55:07 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
2025-11-16 16:55:07 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
2025-11-16 16:55:07 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
2025-11-16 16:55:07 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


## Prepare data

In [ ]:
df = pd.read_csv(data_file_path)
print("Initial number of rows:", len(df))
df = df[df["valid"]]
df = df.dropna()
df = df.drop_duplicates()
assert df.isnull().sum().max() == 0, "DataFrame still contains NaN values after dropping."
print("Number of rows after dropping NaNs:", len(df))
df.head(5)

Initial number of rows: 10000
Number of rows after dropping NaNs: 10000


,section_type,valid,d,b,t_f,t_w,r,n_r,A,ksx,ksy,Ixx,Iyy,Ixy,elastic_modulus,poissons_ratio,yield_strength
0,i_section,True,219.993316,60.082117,16.102008,19.658738,18.206382,8,5919.842002,0.405529,0.673721,3.336576e+07,7.618430e+05,3.759200e-08,142878.634946,0.249836,627.981463
1,i_section,True,218.679892,101.364829,19.045533,10.479593,14.952112,8,5951.381999,0.593593,0.375496,4.522485e+07,3.339757e+06,-7.531825e-08,77794.892054,0.258321,764.354023
2,i_section,True,150.834056,189.261926,20.131272,11.651260,13.691083,8,9074.298663,0.736252,0.179629,3.456668e+07,2.277524e+07,2.738947e-07,117660.413090,0.257656,702.979481
3,i_section,True,153.306185,58.212966,25.862604,20.406344,12.843843,8,5229.919186,0.566963,0.561885,1.451199e+07,9.482954e+05,3.428147e-09,71216.498736,0.283641,765.959260
4,i_section,True,245.654435,244.179242,11.204797,14.114064,19.976091,8,8975.886709,0.570576,0.349767,9.239026e+07,2.729322e+07,-5.592639e-07,80323.213933,0.278583,657.389668


In [6]:
X = df[predictor_columns]
y = df[target_columns]
X.shape, y.shape

((10000, 8), (10000, 6))

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)
print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

Training data shape: (6700, 8)
Testing data shape: (3300, 8)


## Train models

In [8]:
def train_model(
    model: BaseEstimator, 
    params_grid: Optional[dict] = None, 
    model_name: Optional[str] = None
) -> BaseEstimator:
    run = mlflow.start_run(run_name=model_name or model.__class__.__name__)  # --- start main run
    run_id = run.info.run_id
    try:
        # ----- tags & baseline params
        mlflow.set_tag("model_class", model.__class__.__name__)
        mlflow.set_tag("model_name", model_name)
        mlflow.set_tag("stage", "baseline" if not params_grid else "baseline+search")
        mlflow.set_tag("task", task)
        mlflow.set_tag("section_type", section_type)
        mlflow.set_tag("library", "sklearn")

        # ----- fit on train
        model.fit(X_train, y_train.values)
        
        # define cross-validation strategy and log parameters
        kf_params = {"n_splits": 6, "random_state": 42, "shuffle": True}
        kf = KFold(**kf_params)
        logged_kf_params = {f"kf__{k}": v for k, v in kf_params.items()}
        mlflow.log_params(logged_kf_params)

        # ----- hyperparameter search (optional)
        if params_grid:
            mlflow.log_dict(params_grid, "param_grid.json")
            mlflow.start_run(run_name=f"{model_name} - RandomizedSearchCV", nested=True)  #-- start nested run
            try:
                cv = RandomizedSearchCV(
                    estimator=model,
                    param_distributions=params_grid,
                    cv=kf,
                    n_iter=10,
                    n_jobs=-1,
                    random_state=42,
                    refit=True,
                )
                cv.fit(X_train, y_train.values)
                model = cv.best_estimator_
            except Exception as e:
                print(f"Error during hyperparameter search: {e}")
                mlflow.log_text(str(e), "error_log.txt")
                raise e
            finally:
                mlflow.end_run()  # --- end nested run
        
        # log the final model
        mlflow.sklearn.log_model(model, name=model_name, input_example=X_train.iloc[:5])
        
        # log cv scores
        cv_scores = cross_val_score(model, X_train, y_train.values, cv=kf)
        mlflow.log_metric("train_cv_score_mean", float(np.mean(cv_scores)))
        mlflow.log_metric("train_cv_score_std", float(np.std(cv_scores)))
        
        # log regression metrics on test set
        y_true = y_test.values
        y_pred = model.predict(X_test)
        mae = mean_absolute_error(y_true, y_pred)
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_true, y_pred)
        mape = mean_absolute_percentage_error(y_true, y_pred)
        medae = median_absolute_error(y_true, y_pred)
        metrics = {
            "test_mae": mae,
            "test_mse": mse,
            "test_rmse": rmse,
            "test_r2": r2,
            "test_mape": mape,
            "test_medae": medae,
        }
        mlflow.log_metrics(metrics)
        
        # log canonical score - the higher the better
        canonical_score = canonical_regression_score(y_true, y_pred)
        mlflow.log_metric(CANONICAL_SCORE_NAME, canonical_score)
        
        return run_id, model
    except Exception as e:
        print(f"Error during model training: {e}")
        mlflow.log_text(str(e), "error_log.txt")
        raise e
    finally:
        mlflow.end_run()  # --- end main run
        print("MLflow run ended.")

### Baseline model

In [9]:
model_name = "Linear Regression Baseline"
steps = [
    ("scaling", StandardScaler()),
    ("regression", LinearRegression())
]
pipeline = Pipeline(steps)
model = TransformedTargetRegressor(
    regressor=pipeline,
    transformer=StandardScaler()          # scales y (each column independently)
)

run_id, model = train_model(model, model_name=model_name)

MLflow run ended.


### Tuned models

In [11]:
model_name = "Ridge Regression"
steps = [
    ("scaling", StandardScaler()),
    ("regression", Ridge(alpha=0.1))
]
pipeline = Pipeline(steps)
model = TransformedTargetRegressor(
    regressor=pipeline,
    transformer=StandardScaler()          # scales y (each column independently)
)
params_grid = {"regressor__regression__alpha": np.arange(0.0001, 1, 10)}

run_id, model = train_model(model, params_grid=params_grid, model_name=model_name)

/Users/baloghbence/Documents/Projects/demo-steel-beam-cross-section-optimization-ML/.venv/lib/python3.12/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
2025/11/16 16:56:30 INFO mlflow.sklearn.utils: Logging the 5 best runs, no runs will be omitted.
2025/11/16 16:56:32 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: The following failures occurred while performing one or more logging operations: [MlflowException('Failed to perform one or more operations on the run with ID e0afbc909dfd40f8bbcc9949fece8001. Failed operations: [MlflowException(\'Changing param values is not allowed. Params were already logged=\\\'[{\\\'key\\\': \\\'regressor__steps\\\', \\\'old_value\\\': "[(\\\'scaling\\\', StandardScaler()), (\\\'regression\\\', Ridge(alpha=0.1))]", \\\'new_value\\\': "[(\\\'scalin

MLflow run ended.


In [12]:
model_name = "Lasso Regression"
steps = [
    ("scaling", StandardScaler()),
    ("regression", Lasso(alpha=0.1))
]
pipeline = Pipeline(steps)
model = TransformedTargetRegressor(
    regressor=pipeline,
    transformer=StandardScaler()          # scales y (each column independently)
)
params_grid = {"regressor__regression__alpha": np.arange(0.0001, 1, 10)}

run_id, model = train_model(model, params_grid=params_grid, model_name=model_name)

/Users/baloghbence/Documents/Projects/demo-steel-beam-cross-section-optimization-ML/.venv/lib/python3.12/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
2025/11/16 16:57:00 INFO mlflow.sklearn.utils: Logging the 5 best runs, no runs will be omitted.
2025/11/16 16:57:02 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: The following failures occurred while performing one or more logging operations: [MlflowException('Failed to perform one or more operations on the run with ID 88075726654d4199bfb46191715d9367. Failed operations: [MlflowException(\'Changing param values is not allowed. Params were already logged=\\\'[{\\\'key\\\': \\\'regressor__steps\\\', \\\'old_value\\\': "[(\\\'scaling\\\', StandardScaler()), (\\\'regression\\\', Lasso(alpha=0.1))]", \\\'new_value\\\': "[(\\\'scalin

MLflow run ended.


In [13]:
model_name = "Lasso Regression Poly3"
steps = [
    ("feature_eng", PolynomialFeatures(3, include_bias=False)),
    ("scaling", StandardScaler()),
    ("regression", Lasso(alpha=0.1))
]
pipeline = Pipeline(steps)
model = TransformedTargetRegressor(
    regressor=pipeline,
    transformer=StandardScaler()          # scales y (each column independently)
)
params_grid = {"regressor__regression__alpha": np.arange(0.0001, 1, 10)}

run_id, model = train_model(model, params_grid=params_grid, model_name=model_name)

/Users/baloghbence/Documents/Projects/demo-steel-beam-cross-section-optimization-ML/.venv/lib/python3.12/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/Users/baloghbence/Documents/Projects/demo-steel-beam-cross-section-optimization-ML/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.818e-01, tolerance: 5.584e-01
  model = cd_fast.enet_coordinate_descent(
/Users/baloghbence/Documents/Projects/demo-steel-beam-cross-section-optimization-ML/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of it

MLflow run ended.


In [14]:
model_name = "Ridge Regression Poly3"
steps = [
    ("feature_eng", PolynomialFeatures(3, include_bias=False)),
    ("scaling", StandardScaler()),
    ("regression", Ridge(alpha=0.1))
]
pipeline = Pipeline(steps)
model = TransformedTargetRegressor(
    regressor=pipeline,
    transformer=StandardScaler()          # scales y (each column independently)
)
params_grid = {"regressor__regression__alpha": np.arange(0.0001, 1, 10)}

run_id, model = train_model(model, params_grid=params_grid, model_name=model_name)

/Users/baloghbence/Documents/Projects/demo-steel-beam-cross-section-optimization-ML/.venv/lib/python3.12/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
2025/11/16 16:57:33 INFO mlflow.sklearn.utils: Logging the 5 best runs, no runs will be omitted.
2025/11/16 16:57:35 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: The following failures occurred while performing one or more logging operations: [MlflowException('Failed to perform one or more operations on the run with ID edd5c60c939740848dc0f3b7966a0585. Failed operations: [MlflowException(\'Changing param values is not allowed. Params were already logged=\\\'[{\\\'key\\\': \\\'regressor__steps\\\', \\\'old_value\\\': "[(\\\'feature_eng\\\', PolynomialFeatures(degree=3, include_bias=False)), (\\\'scaling\\\', StandardScaler()), (

MLflow run ended.
